In [1]:
from scipy.spatial import distance as dist
from imutils.video import VideoStream
from imutils import face_utils
from threading import Thread, Lock  
import numpy as np
import pyglet
import argparse
import imutils
import time
import dlib
import cv2
import sys 
import os
from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity

# --- GLOBAL STATE AND LOCKS ---
ALARM_LOCK = Lock()
PAUSED = False
ALARM_ON = False
AUTHENTICATED = False 
AUTHENTICATION_TIMEOUT_FRAMES = 150 
AUTHENTICATION_FRAME_COUNT = 0

# Global variable to store the name of the authorized driver
CURRENT_DRIVER_NAME = "Unauthenticated"
# --- END GLOBAL STATE ---

# --- CONSTANTS ---
EYE_AR_THRESH = 0.26
EYE_AR_CONSEC_FRAMES = 30
PULLED_OVER_FRAMES = 170 # Drowsiness critical duration (~10 seconds)
FACE_LOST_ALARM_FRAMES = 60 # Face lost warning alarm duration (~2 seconds)

# NEW: Critical duration for face lost before pull-over (~20 seconds)
FACE_NOT_DETECTED_CRITICAL_PULLOVER = 200

COUNTER = 0
FINAL_COUNTER = 0 
FACE_NOT_DETECTED_COUNTER = 0
# --- END CONSTANTS ---

# --- ARC FACE & DB SETUP ---
print("🔄 Loading ArcFace model (InsightFace for Authentication)...")
FA_APP = FaceAnalysis(name="buffalo_l", providers=['CPUExecutionProvider'])
FA_APP.prepare(ctx_id=0)
print("✅ ArcFace ready!")

FACE_DB = {}
AUTH_THRESHOLD = 0.50 

def load_known_faces():
    global FACE_DB
    FACE_DB = {}
    known_faces_path = "known_faces"
    os.makedirs(known_faces_path, exist_ok=True)
    
    for file in os.listdir(known_faces_path):
        if file.endswith(".jpg") or file.endswith(".png"):
            name = os.path.splitext(file)[0]
            img = cv2.imread(os.path.join(known_faces_path, file))
            if img is None:
                print(f"[ERROR] Could not read image file: {file}")
                continue
                
            faces = FA_APP.get(img)
            if faces:
                FACE_DB[name] = faces[0].normed_embedding
    print(f"📁 Loaded {len(FACE_DB)} known face(s) for authentication.")

load_known_faces()
# --- END ARC FACE & DB SETUP ---


def sound_alarm(path):
    try:
        music = pyglet.resource.media(path)
        music.play()
        pyglet.app.run()
    except pyglet.resource.ResourceNotFoundException:
        print(f"[ERROR] Alarm file not found: {path}")
    except Exception as e:
        pass

def eye_aspect_ratio(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    ear = (A + B) / (2.0 * C)
    return ear

# --- ARGPARSE & INITIALIZATION (Dlib DDS) ---
ap = argparse.ArgumentParser()
ap.add_argument("-w", "--webcam", type=int, default=0, help="index of webcam on system")
ap.add_argument("-a", "--alarm", type=str, default="alarm.wav", help="path to alarm .WAV file")
args, unknown = ap.parse_known_args()
args = vars(args)

print("[INFO] loading facial landmark predictor (Dlib for DDS)...")
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("68 face landmarks.dat")

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
(lStart, lEnd) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]
(rStart, rEnd) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]

print("[INFO] starting video stream thread...")
vs = VideoStream(src=args["webcam"]).start()
time.sleep(1.0)
# --- END ARGPARSE & INITIALIZATION ---


# --- MAIN APPLICATION LOOP ---
while True:
    frame = vs.read()
    if frame is None:
        break

    frame = imutils.resize(frame, width=450)
    frame_display = frame.copy() 

    # =========================================================================
    # PHASE 1: AUTHENTICATION GATE (LOCKOUT STATE)
    # =========================================================================
    if not AUTHENTICATED:
        
        cv2.putText(frame_display, "SYSTEM LOCKED - AUTHENTICATION REQUIRED", (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.putText(frame_display, f"Attempting match... ({AUTHENTICATION_FRAME_COUNT})", (10, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        faces_fa = FA_APP.get(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        authenticated_driver_name = None
        
        for face in faces_fa:
            bbox = face.bbox.astype(int)
            emb = face.normed_embedding
            cv2.rectangle(frame_display, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 255), 2)
            max_sim = 0.0
            
            for db_name, db_emb in FACE_DB.items():
                sim = cosine_similarity([emb], [db_emb])[0][0]
                
                if sim > AUTH_THRESHOLD and sim > max_sim:
                    max_sim = sim
                    authenticated_driver_name = db_name
            
            if authenticated_driver_name:
                # --- CAPTURE DRIVER NAME ON SUCCESS ---
                global CURRENT_DRIVER_NAME
                CURRENT_DRIVER_NAME = authenticated_driver_name 
                # --------------------------------------

                cv2.putText(frame_display, f"MATCH: {authenticated_driver_name} ({max_sim:.2f})", 
                             (bbox[0], bbox[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                break
            else:
                 cv2.putText(frame_display, f"UNKNOWN ({max_sim:.2f})", 
                             (bbox[0], bbox[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)


        if authenticated_driver_name:
            AUTHENTICATED = True
            print(f"\n[AUTH] Driver '{CURRENT_DRIVER_NAME}' authenticated successfully. DDS ACTIVE.")
        
        
        # Handle timeout if no one is authenticated within time limit
        AUTHENTICATION_FRAME_COUNT += 1
        if AUTHENTICATION_FRAME_COUNT >= AUTHENTICATION_TIMEOUT_FRAMES and not AUTHENTICATED:
             cv2.putText(frame_display, "AUTHENTICATION FAILED/TIMEOUT. SYSTEM LOCKED.", (10, 90),
                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
             cv2.imshow("Frame", frame_display)
             key = cv2.waitKey(0) & 0xFF
             if key == ord("q"): break
             if key == ord("r"): AUTHENTICATION_FRAME_COUNT = 0
             
        
        if not AUTHENTICATED:
            cv2.imshow("Frame", frame_display)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"): break
            continue 

    # =========================================================================
    # PHASE 2: DRIVER DROWSINESS DETECTION (DDS) - Only runs if AUTHENTICATED
    # =========================================================================

    # 1. PAUSED State Check (DDS Intervention)
    if PAUSED:
        frame_paused = np.zeros((450, 450, 3), dtype="uint8")
        cv2.putText(frame_paused, "CAR PULLED OVER. ADMIN CONTACTED.", (10, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.putText(frame_paused, "PRESS 'R' TO RE-ENABLE MONITORING.", (10, 100),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow("Frame", frame_paused)
        
        key = cv2.waitKey(0) & 0xFF
        if key == ord("r"):
            with ALARM_LOCK:
                PAUSED = False
                COUNTER = 0
                FINAL_COUNTER = 0
                ALARM_ON = False
                FACE_NOT_DETECTED_COUNTER = 0 
            
            # --- CRITICAL RESET FOR RE-AUTHENTICATION ---
            CURRENT_DRIVER_NAME = "Unauthenticated" 
            AUTHENTICATED = False 
            # --------------------------------------------
            print("[INFO] System resumed. Re-authentication required.")
        elif key == ord("q"): break
        continue
        
    # 2. Preparation for Dlib DDS
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    enhanced_gray = clahe.apply(gray)
    rects = detector(enhanced_gray, 1)

    # --- Face Not Detected Logic (Warning and Critical Pull-over) ---
    if len(rects) == 0:
        FACE_NOT_DETECTED_COUNTER += 1
        
        # Check if face is lost long enough to trigger the WARNING ALARM (~2s)
        if FACE_NOT_DETECTED_COUNTER >= FACE_LOST_ALARM_FRAMES:
            # 1. Fire alarm if not already active
            with ALARM_LOCK:
                if not ALARM_ON:
                    ALARM_ON = True
                    t = Thread(target=sound_alarm, args=(args["alarm"],))
                    t.deamon = True
                    t.start()
            
            # 2. Display ALARM message
            cv2.putText(frame_display, "ALARM! DRIVER NOT IN FRAME!", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            cv2.putText(frame_display, f"Cont. Lost: {FACE_NOT_DETECTED_COUNTER}/{FACE_NOT_DETECTED_CRITICAL_PULLOVER}", (10, 60),
                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        else:
             cv2.putText(frame_display, "DRIVER OUT OF FRAME!", (10, 30),
                 cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # 3. CRITICAL PULL-OVER CHECK (After 20 seconds)
        if FACE_NOT_DETECTED_COUNTER >= FACE_NOT_DETECTED_CRITICAL_PULLOVER:
            print(f"[ALERT] Driver out of frame for critical duration ({FACE_NOT_DETECTED_CRITICAL_PULLOVER} frames). Car stopped.")
            with ALARM_LOCK:
                ALARM_ON = False 
                PAUSED = True
            continue 

        # If face is lost, reset drowsiness counters (since we can't measure EAR)
        COUNTER = 0
        FINAL_COUNTER = 0

    else:
        # Face IS detected, reset face lost counter
        FACE_NOT_DETECTED_COUNTER = 0

        # If alarm was active due to face lost, reset it now that the face is back
        with ALARM_LOCK:
             if ALARM_ON and COUNTER == 0 and FINAL_COUNTER == 0:
                 ALARM_ON = False
                 print("[INFO] Alarm reset (Face found).")
                 
    # 3. Drowsiness Monitoring Loop
    for rect in rects:
        # --- Draw face outline ---
        (x, y, w, h) = face_utils.rect_to_bb(rect)
        cv2.rectangle(frame_display, (x, y), (x + w, y + h), (255, 0, 0), 2) # Blue rectangle for face
        # --- End Draw face outline ---

        # Determine the facial landmarks for the face region
        shape = predictor(gray, rect)
        shape = face_utils.shape_to_np(shape)

        # extract the left and right eye coordinates and compute EAR
        leftEye = shape[lStart:lEnd]
        rightEye = shape[rStart:rEnd]
        leftEAR = eye_aspect_ratio(leftEye)
        rightEAR = eye_aspect_ratio(rightEye)
        ear = (leftEAR + rightEAR) / 2.0

        cv2.drawContours(frame_display, [cv2.convexHull(leftEye)], -1, (0, 255, 0), 1)
        cv2.drawContours(frame_display, [cv2.convexHull(rightEye)], -1, (0, 255, 0), 1)

        # check to see if the eye aspect ratio is below the threshold
        if ear < EYE_AR_THRESH:
            COUNTER += 1
            FINAL_COUNTER += 1 

            # --- INITIAL DROWSINESS ALARM ---
            if COUNTER >= EYE_AR_CONSEC_FRAMES:
                with ALARM_LOCK:
                    if not ALARM_ON:
                        ALARM_ON = True
                        t = Thread(target=sound_alarm, args=(args["alarm"],))
                        t.deamon = True
                        t.start()
                cv2.putText(frame_display, "DROWSINESS ALERT!", (10, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            # --- CRITICAL DROWSINESS PULL-OVER ---
            if FINAL_COUNTER >= PULLED_OVER_FRAMES:
                print("[ALERT] Drowsiness exceeded 10 seconds. Car stopped.")
                with ALARM_LOCK:
                    ALARM_ON = False
                    PAUSED = True
                continue

        # otherwise, the eye aspect ratio is not below the blink threshold
        else:
            COUNTER = 0
            FINAL_COUNTER = 0
            with ALARM_LOCK:
                if ALARM_ON and FACE_NOT_DETECTED_COUNTER == 0:
                    ALARM_ON = False
                    print("[INFO] Drowsiness alarm reset.")


        # --- DRAW DRIVER NAME ---
        # NOTE: Position changed to be in the left top corner (10, 30) for clear display
        cv2.putText(frame_display, f"Driver: {CURRENT_DRIVER_NAME}", (10, 20),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2) # Yellow color
        # ------------------------

        # Draw HUD details (DDS data displayed in top right)
        cv2.putText(frame_display, "EAR: {:.2f}".format(ear), (300, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.putText(frame_display, "Cont. Drowsy Frames: {}".format(FINAL_COUNTER), (150, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

    # Final display
    cv2.imshow("Frame", frame_display)
    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break

# --- CLEANUP ---
print("[INFO] cleaning up...")
cv2.destroyAllWindows()
vs.stop()

C:\Users\santh\anaconda3\Lib\site-packages\albumentations\check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno 11001] getaddrinfo failed>
  data = fetch_version_info()


🔄 Loading ArcFace model (InsightFace for Authentication)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\santh/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\santh/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\santh/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\santh/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\santh/.insightface\models\buf

C:\Users\santh\anaconda3\Lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


📁 Loaded 6 known face(s) for authentication.
[INFO] loading facial landmark predictor (Dlib for DDS)...
[INFO] starting video stream thread...

[AUTH] Driver 'Santhosh' authenticated successfully. DDS ACTIVE.
[INFO] Drowsiness alarm reset.
[INFO] Drowsiness alarm reset.
[INFO] cleaning up...
